# Model Comparison: Random Forest vs Custom CNN
### 204466 Deep Learning — Final Project

**Before running this notebook:**
1. Run `baseline_random_forest.ipynb` → generates `rf_results.json`
2. Run `skin_lesion_classification.ipynb` → generates `cnn_results.json`

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Load results from both models
with open('rf_results.json',  'r') as f:
    rf_results  = json.load(f)

with open('cnn_results.json', 'r') as f:
    cnn_results = json.load(f)

print("Results loaded successfully")
print(f"  Random Forest — Accuracy: {rf_results['accuracy']:.4f}")
print(f"  Custom CNN    — Accuracy: {cnn_results['accuracy']:.4f}")

## 1. Overall Metrics Comparison

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1']
rf_scores  = [rf_results[m]  for m in metrics]
cnn_scores = [cnn_results[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, rf_scores,  width, label='Random Forest', color='#f4a261', edgecolor='white')
bars2 = ax.bar(x + width/2, cnn_scores, width, label='Custom CNN',    color='#457b9d', edgecolor='white')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Overall Metrics: Random Forest vs Custom CNN')
ax.set_xticks(x)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1-Score'])
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('comparison_overall.png', dpi=150)
plt.show()

## 2. Per-Class F1 Score Comparison

In [ ]:
class_names = list(rf_results['per_class_f1'].keys())
rf_f1  = [rf_results['per_class_f1'][c]  for c in class_names]
cnn_f1 = [cnn_results['per_class_f1'][c] for c in class_names]

x = np.arange(len(class_names))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, rf_f1,  width, label='Random Forest', color='#f4a261', edgecolor='white')
ax.bar(x + width/2, cnn_f1, width, label='Custom CNN',    color='#457b9d', edgecolor='white')

ax.set_ylim(0, 1.1)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score: Random Forest vs Custom CNN')
ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_per_class.png', dpi=150)
plt.show()

## 3. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric':        ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1 (weighted)'],
    'Random Forest': [f"{rf_results['accuracy']:.4f}",
                      f"{rf_results['precision']:.4f}",
                      f"{rf_results['recall']:.4f}",
                      f"{rf_results['f1']:.4f}"],
    'Custom CNN':    [f"{cnn_results['accuracy']:.4f}",
                      f"{cnn_results['precision']:.4f}",
                      f"{cnn_results['recall']:.4f}",
                      f"{cnn_results['f1']:.4f}"],
})

print(summary.to_string(index=False))

diff_acc = cnn_results['accuracy'] - rf_results['accuracy']
print(f"\nCNN improves over Random Forest by: {diff_acc*100:+.2f}% accuracy")

## 4. Why CNN outperforms Random Forest?

| | Random Forest + HOG | Custom CNN |
|---|---|---|
| **Feature extraction** | Manual (HOG) | Automatic (learned) |
| **Spatial understanding** | Limited | Full (via convolutions) |
| **Training data needed** | Low | High |
| **Training time** | Fast (minutes) | Slow (hours) |
| **Accuracy on complex images** | Lower | Higher |
| **Handles class imbalance** | class_weight='balanced' | WeightedSampler + Weighted Loss |